# SSB Config Admin

Komplett notebook for å administrere `ssb_config`-tabellen.

## 📋 Oversikt

**Bruk:** Sett parametere i neste celle og kjør hele notebooken (Run All).
Steg som har `None`-parametere hoppes automatisk over.

### Steg:
1. **Se alle tabeller** – Oversikt over config
2. **Legg til ny tabell** – Enkeltinnlegg (henter metadata fra SSB API)
3. **Oppdater tabell** – Endre kategori / lookback_periods / priority
4. **Slett tabell** – Fjern fra config
5. **Vis loading-historikk** – Når ble tabeller sist lastet?
6. **Mark as loaded** – Utility-funksjon (bruk i ingest-script)
7. **Reset timestamp** – Force re-load neste gang Update Detector kjører
8. **Reset til default** – Batch-reset av kategori/lookback til API-verdier
9. **Batch insert** – Legg til flere tabeller på én gang
10. **Reset alle** – Alle tabeller til default (krever confirm=True)

### ssb_config-schema:
| Kolonne | Type | Beskrivelse |
|---|---|---|
| `table_id` | STRING | SSB tabellnummer, PK |
| `table_name` | STRING | Tabellnavn fra SSB API |
| `frequency` | STRING | Annual / Quarterly / Monthly / Weekly / Daily |
| `category` | STRING | Tematisk kategori fra SSB |
| `lookback_periods` | INT | Antall perioder å hente ved oppdatering |
| `priority` | STRING | CRITICAL eller NORMAL (default: NORMAL) |
| `last_loaded_timestamp` | STRING | ISO-timestamp for siste vellykkede last |

---


In [1]:
# ============================================================================
# IMPORTS
# ============================================================================

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from delta.tables import DeltaTable
import requests
from datetime import datetime

spark = SparkSession.builder.appName("SSBConfigAdmin").getOrCreate()

CONFIG_TABLE = "pipeline.ssb_config"

print("✅ Imports lastet")
print(f"   Konfig-tabell: {CONFIG_TABLE}")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 3, Finished, Available, Finished, False)

✅ Imports lastet
   Konfig-tabell: pipeline.ssb_config


In [2]:
# ============================================================================
# PARAMETRE – Endre disse verdiene, kjør deretter Run All
# Steg der parameteren er None hoppes automatisk over.
# ============================================================================

# --- LEGG TIL NY TABELL (enkelt) ---
tabell_id        = "12306"    # eks: "14306"
kategori         = None    # Override kategori fra API  (None = bruk API-verdi)
lookback         = None    # Override lookback_periods  (None = bruk default)
priority_ny      = None    # "CRITICAL" eller "NORMAL"  (None = "NORMAL")

# --- LEGG TIL FLERE TABELLER (batch) ---
table_list       = []      # eks: ["14305", "14307"]

# --- OPPDATER EKSISTERENDE TABELL ---
tabell_id_update = None    # eks: "01222"
ny_kategori      = None    # Ny kategori       (None = hent default fra API)
ny_lookback      = None    # Ny lookback       (None = hent default fra API)
ny_priority      = None    # "CRITICAL" / "NORMAL" (None = ingen endring)

# --- SLETT TABELL ---
tabell_id_delete = None    # eks: "09429"

# --- RESET TIMESTAMP (force re-load) ---
tabell_id_reset  = "07459"    # eks: "07459"

# --- RESET TIL DEFAULT (batch fra liste) ---
table_list_default = []    # eks: ["14305", "14307"]

print("✅ Parametre definert")
print(f"   tabell_id:          {tabell_id}")
print(f"   table_list:         {table_list}")
print(f"   tabell_id_update:   {tabell_id_update}")
print(f"   tabell_id_delete:   {tabell_id_delete}")
print(f"   tabell_id_reset:    {tabell_id_reset}")
print(f"   table_list_default: {table_list_default}")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 4, Finished, Available, Finished, False)

✅ Parametre definert
   tabell_id:          12306
   table_list:         []
   tabell_id_update:   None
   tabell_id_delete:   None
   tabell_id_reset:    07459
   table_list_default: []


In [3]:
# ============================================================================
# FUNKSJONER
# ============================================================================

VALID_PRIORITIES = {"CRITICAL", "NORMAL"}

SSB_CONFIG_SCHEMA = StructType([
    StructField("table_id",              StringType(),  False),
    StructField("table_name",            StringType(),  True),
    StructField("frequency",             StringType(),  True),
    StructField("category",              StringType(),  True),
    StructField("lookback_periods",      IntegerType(), True),
    StructField("priority",              StringType(),  True),
    StructField("last_loaded_timestamp", StringType(),  True),
])


def _ensure_priority_column():
    """
    Sikrer at ssb_config har priority-kolonnen.
    Trygg å kalle flere ganger – gjør ingenting hvis kolonnen allerede finnes.
    """
    try:
        cols = [f.name for f in spark.table(CONFIG_TABLE).schema.fields]
        if "priority" not in cols:
            print("⚙️  Legger til manglende 'priority'-kolonne i ssb_config...")
            spark.sql(f"""
                ALTER TABLE {CONFIG_TABLE}
                ADD COLUMN priority STRING
            """)
            spark.sql(f"""
                UPDATE {CONFIG_TABLE}
                SET priority = 'NORMAL'
                WHERE priority IS NULL
            """)
            print("✅ 'priority'-kolonne lagt til og satt til NORMAL for alle rader")
    except Exception:
        pass  # tabellen finnes kanskje ikke ennå – ok


def fetch_ssb_table_info(table_id: str) -> dict:
    """Hent grunnleggende tabell-info fra SSB PXWeb v2 API"""
    url = f"https://data.ssb.no/api/pxwebapi/v2/tables/{table_id}"
    try:
        response = requests.get(
            url,
            params={"lang": "no"},
            headers={"Accept": "application/json", "Accept-Language": "no"},
            timeout=15,
        )
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            raise ValueError(f"❌ Tabell {table_id} finnes IKKE på SSB API")
        raise ValueError(f"❌ HTTP-feil for tabell {table_id}: {e}")
    except requests.exceptions.RequestException as e:
        raise ValueError(f"❌ Nettverksfeil for tabell {table_id}: {e}")

    data = response.json()

    category = None
    paths = data.get("paths", [])
    if paths and len(paths[0]) > 0:
        category = paths[0][0].get("label")

    return {
        "table_id":              data.get("id", table_id),
        "table_name":            data.get("label", "Ukjent"),
        "frequency":             data.get("timeUnit", "Unknown"),
        "category":              category,
        "lookback_periods":      None,
        "priority":              "NORMAL",
        "last_loaded_timestamp": None,
    }


def get_default_lookback(frequency: str) -> int:
    """Standard lookback-perioder basert på frekvens"""
    return {
        "Annual":    2,
        "Yearly":    2,
        "Quarterly": 4,
        "Monthly":   12,
        "Weekly":    52,
        "Daily":     365,
    }.get(frequency, 2)


def _table_exists_in_config(table_id: str) -> bool:
    """Sjekk om table_id allerede er i config (parameterisert query)"""
    count = (
        spark.table(CONFIG_TABLE)
        .filter(F.col("table_id") == table_id)
        .count()
    )
    return count > 0


def add_new_table(
    table_id: str,
    override_category: str = None,
    override_lookback: int = None,
    override_priority: str = None,
) -> bool:
    """
    Legg til ny tabell i config (henter metadata fra SSB API).

    Returns:
        True hvis lagt til, False hvis allerede finnes eller feil.
    """
    _ensure_priority_column()

    if _table_exists_in_config(table_id):
        print(f"⚠️  Tabell {table_id} finnes allerede i config – hopper over")
        return False

    print(f"📡 Henter info for tabell {table_id} fra SSB API...")
    try:
        info = fetch_ssb_table_info(table_id)
    except Exception as e:
        print(str(e))
        return False

    info["lookback_periods"] = (
        override_lookback
        if override_lookback is not None
        else get_default_lookback(info["frequency"])
    )
    if override_category is not None:
        info["category"] = override_category

    raw_priority = (override_priority or "NORMAL").upper()
    info["priority"] = raw_priority if raw_priority in VALID_PRIORITIES else "NORMAL"

    print(f"\n✅ Klar til lagring:")
    for k, v in info.items():
        print(f"   {k}: {v}")

    new_df = spark.createDataFrame([info], schema=SSB_CONFIG_SCHEMA)
    new_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(CONFIG_TABLE)
    print(f"\n✅ Tabell {table_id} lagt til i config!")
    return True


def reset_table_to_defaults(table_id: str) -> bool:
    """
    Reset kategori og lookback_periods til ferske verdier fra SSB API.
    last_loaded_timestamp og priority røres ikke.
    """
    _ensure_priority_column()

    if not _table_exists_in_config(table_id):
        print(f"❌ Tabell {table_id} finnes IKKE i config!")
        return False

    print(f"📡 Henter frisk info fra SSB API for {table_id}...")
    try:
        info = fetch_ssb_table_info(table_id)
    except Exception as e:
        print(str(e))
        return False

    default_lookback = get_default_lookback(info["frequency"])
    new_category     = info["category"] or ""

    (
        DeltaTable.forName(spark, CONFIG_TABLE)
        .update(
            condition=F.col("table_id") == table_id,
            set={
                "category":         F.lit(new_category),
                "lookback_periods": F.lit(default_lookback),
            },
        )
    )

    print(f"✅ Tabell {table_id} resettet til default-verdier")
    print(f"   category         → {new_category}")
    print(f"   lookback_periods → {default_lookback}")
    display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == table_id))
    return True


def mark_table_as_loaded(table_id: str):
    """
    Marker tabell som lastet – oppdaterer last_loaded_timestamp til nå.
    Kall denne fra ingest-script etter vellykket last:
        mark_table_as_loaded(table_id)
    """
    timestamp = datetime.now().isoformat()
    (
        DeltaTable.forName(spark, CONFIG_TABLE)
        .update(
            condition=F.col("table_id") == table_id,
            set={"last_loaded_timestamp": F.lit(timestamp)},
        )
    )
    print(f"✅ Tabell {table_id} merket som lastet ({timestamp})")


def reset_all_tables_to_defaults(confirm: bool = False):
    """
    Reset ALLE tabeller i config til default-verdier fra SSB API.
    Args:
        confirm: Må settes til True for å kjøre.
    """
    if not confirm:
        print("⚠️  ADVARSEL: Denne funksjonen resetter ALLE tabeller!")
        print("   For å kjøre: reset_all_tables_to_defaults(confirm=True)")
        return

    table_ids = [
        row["table_id"]
        for row in spark.table(CONFIG_TABLE).select("table_id").orderBy("table_id").collect()
    ]

    if not table_ids:
        print("⚠️  Ingen tabeller funnet i config!")
        return

    print(f"📋 Resetter {len(table_ids)} tabeller: {', '.join(table_ids)}\n")
    ok = err = 0
    for tid in table_ids:
        print(f"--- {tid} ---")
        ok += reset_table_to_defaults(tid)
        err += not reset_table_to_defaults(tid)
        print()

    print(f"📊 Ferdig: {ok} OK, {err} feil")


print("✅ Funksjoner definert")
print(f"   Kjører migrering av priority-kolonne hvis nødvendig...")
_ensure_priority_column()

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 5, Finished, Available, Finished, False)

✅ Funksjoner definert
   Kjører migrering av priority-kolonne hvis nødvendig...


In [4]:
# ============================================================================
# 1  SE ALLE TABELLER I CONFIG
# ============================================================================

print("=" * 80)
print("1  SE ALLE TABELLER I CONFIG")
print("=" * 80)

if not spark.catalog.tableExists(CONFIG_TABLE):
    print(f"\nTabellen '{CONFIG_TABLE}' finnes ikke enda.")
    print("Kjor steg 9 (Batch insert) for aa opprette den forste gang.\n")
else:
    config_df = spark.sql(f"""
        SELECT
            table_id,
            table_name,
            frequency,
            category,
            lookback_periods,
            COALESCE(priority, 'NORMAL') AS priority,
            last_loaded_timestamp,
            CASE
                WHEN last_loaded_timestamp IS NULL THEN 'Aldri lastet'
                ELSE last_loaded_timestamp
            END AS status
        FROM {CONFIG_TABLE}
        ORDER BY priority DESC, category, table_name
    """)
    print(f"\nTotalt {config_df.count()} tabeller i config:\n")
    display(config_df)


StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 6, Finished, Available, Finished, False)

1  SE ALLE TABELLER I CONFIG

Totalt 7 tabeller i config:



SynapseWidget(Synapse.DataFrame, 11dc655e-b6b5-4800-bf24-0f3327cfe513)

In [5]:
# ============================================================================
# 2️⃣  LEGG TIL NY TABELL (enkelt)
# ============================================================================

print("=" * 80)
print("2️⃣  LEGG TIL NY TABELL")
print("=" * 80)

if tabell_id is None:
    print("\n⏭️  Hoppet over – tabell_id er None")
    print("💡 Sett tabell_id = 'XXXXX' i parametercellen for å legge til en tabell")
else:
    add_new_table(
        tabell_id,
        override_category=kategori,
        override_lookback=lookback,
        override_priority=priority_ny,
    )

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 7, Finished, Available, Finished, False)

2️⃣  LEGG TIL NY TABELL
⚠️  Tabell 12306 finnes allerede i config – hopper over


In [6]:
# ============================================================================
# 3️⃣  OPPDATER EKSISTERENDE TABELL
# ============================================================================

print("=" * 80)
print("3️⃣  OPPDATER EKSISTERENDE TABELL")
print("=" * 80)

if tabell_id_update is None:
    print("\n⏭️  Hoppet over – tabell_id_update er None")
else:
    print(f"\n🔄 Oppdaterer tabell {tabell_id_update}...")
    _ensure_priority_column()

    if not _table_exists_in_config(tabell_id_update):
        print(f"❌ Tabell {tabell_id_update} finnes IKKE i config!")
    else:
        # Hent default-verdier fra API for felter som er None
        final_kategori = ny_kategori
        final_lookback = ny_lookback

        if ny_kategori is None or ny_lookback is None:
            print(f"📡 Henter default-verdier fra SSB API...")
            try:
                info = fetch_ssb_table_info(tabell_id_update)
                if ny_kategori is None:
                    final_kategori = info["category"]
                    print(f"   ℹ️  category → {final_kategori} (fra API)")
                if ny_lookback is None:
                    final_lookback = get_default_lookback(info["frequency"])
                    print(f"   ℹ️  lookback_periods → {final_lookback} (default for {info['frequency']})")
            except Exception as e:
                print(f"❌ Feil ved henting fra SSB API: {e}")
                print("   Avbryter oppdatering – sett ny_kategori og ny_lookback manuelt")
                final_kategori = None

        if final_kategori is not None:
            set_dict = {
                "category":         F.lit(final_kategori),
                "lookback_periods": F.lit(int(final_lookback)),
            }
            if ny_priority is not None:
                valid_prio = ny_priority.upper() if ny_priority.upper() in VALID_PRIORITIES else "NORMAL"
                set_dict["priority"] = F.lit(valid_prio)
                print(f"   ℹ️  priority → {valid_prio}")

            DeltaTable.forName(spark, CONFIG_TABLE).update(
                condition=F.col("table_id") == tabell_id_update,
                set=set_dict,
            )

            print(f"\n✅ Tabell {tabell_id_update} oppdatert!")
            display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == tabell_id_update))

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 8, Finished, Available, Finished, False)

3️⃣  OPPDATER EKSISTERENDE TABELL

⏭️  Hoppet over – tabell_id_update er None


In [7]:
# ============================================================================
# 4️⃣  SLETT EN TABELL
# ============================================================================

print("=" * 80)
print("4️⃣  SLETT EN TABELL")
print("=" * 80)

if tabell_id_delete is None:
    print("\n⏭️  Hoppet over – tabell_id_delete er None")
else:
    print(f"\n🗑️  Sletter tabell {tabell_id_delete}...")

    if not _table_exists_in_config(tabell_id_delete):
        print(f"❌ Tabell {tabell_id_delete} finnes IKKE i config!")
    else:
        print("📋 Rad som slettes:")
        display(spark.table(CONFIG_TABLE).filter(F.col("table_id") == tabell_id_delete))

        DeltaTable.forName(spark, CONFIG_TABLE).delete(
            condition=F.col("table_id") == tabell_id_delete
        )
        print(f"\n✅ Tabell {tabell_id_delete} slettet fra config!")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 9, Finished, Available, Finished, False)

4️⃣  SLETT EN TABELL

⏭️  Hoppet over – tabell_id_delete er None


In [8]:
# ============================================================================
# 5  VIS LOADING-HISTORIKK
# ============================================================================

print("=" * 80)
print("5  LOADING-HISTORIKK")
print("=" * 80)

if not spark.catalog.tableExists(CONFIG_TABLE):
    print(f"\nTabellen '{CONFIG_TABLE}' finnes ikke enda.")
else:
    historikk_df = spark.sql(f"""
        SELECT
            table_id,
            table_name,
            frequency,
            COALESCE(priority, 'NORMAL') AS priority,
            last_loaded_timestamp,
            CASE
                WHEN last_loaded_timestamp IS NULL THEN 'ALDRI LASTET'
                ELSE last_loaded_timestamp
            END AS loading_status
        FROM {CONFIG_TABLE}
        ORDER BY
            CASE WHEN last_loaded_timestamp IS NULL THEN 0 ELSE 1 END,
            last_loaded_timestamp DESC
    """)
    print(f"\nLoading-status for alle tabeller:\n")
    display(historikk_df)


StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 10, Finished, Available, Finished, False)

5  LOADING-HISTORIKK

Loading-status for alle tabeller:



SynapseWidget(Synapse.DataFrame, 03b5dea4-91ec-4cca-adf9-df99e549fe63)

In [9]:
# ============================================================================
# 6️⃣  UTILITY: Marker tabell som lastet
# ============================================================================

print("=" * 80)
print("6️⃣  UTILITY: Marker tabell som lastet")
print("=" * 80)

print("""
💡 Kall denne funksjonen fra ingest-script etter vellykket last:

    mark_table_as_loaded(table_id)

Eksempel:
    # Etter at data er skrevet til Lakehouse:
    mark_table_as_loaded("07459")

# Kommenter inn for å teste:
# mark_table_as_loaded("12345")
""")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 11, Finished, Available, Finished, False)

6️⃣  UTILITY: Marker tabell som lastet

💡 Kall denne funksjonen fra ingest-script etter vellykket last:

    mark_table_as_loaded(table_id)

Eksempel:
    # Etter at data er skrevet til Lakehouse:
    mark_table_as_loaded("07459")

# Kommenter inn for å teste:
# mark_table_as_loaded("12345")



In [10]:
# ============================================================================
# 7️⃣  RESET TIMESTAMP (force re-load neste kjøring)
# ============================================================================

print("=" * 80)
print("7️⃣  RESET TIMESTAMP")
print("=" * 80)

if tabell_id_reset is None:
    print("\n⏭️  Hoppet over – tabell_id_reset er None")
else:
    print(f"\n🔄 Resetter last_loaded_timestamp for {tabell_id_reset}...")

    if not _table_exists_in_config(tabell_id_reset):
        print(f"❌ Tabell {tabell_id_reset} finnes IKKE i config!")
    else:
        DeltaTable.forName(spark, CONFIG_TABLE).update(
            condition=F.col("table_id") == tabell_id_reset,
            set={"last_loaded_timestamp": F.lit(None).cast(StringType())},
        )
        print(f"✅ {tabell_id_reset}: last_loaded_timestamp → NULL")
        print(f"   Tabellen vil bli lastet som 'first_load' neste gang Update Detector kjører")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 12, Finished, Available, Finished, False)

7️⃣  RESET TIMESTAMP

🔄 Resetter last_loaded_timestamp for 07459...
✅ 07459: last_loaded_timestamp → NULL
   Tabellen vil bli lastet som 'first_load' neste gang Update Detector kjører


In [11]:
# ============================================================================
# 8️⃣  RESET TIL DEFAULT (batch fra liste)
# ============================================================================

print("=" * 80)
print("8️⃣  RESET TIL DEFAULT")
print("=" * 80)

if not table_list_default:
    print("\n⏭️  Hoppet over – table_list_default er tom")
    print("💡 Sett table_list_default = ['12345', '67890'] for å resette")
else:
    print(f"\n📋 Resetter {len(table_list_default)} tabeller til default...\n")
    for tid in table_list_default:
        print(f"--- {tid} ---")
        reset_table_to_defaults(tid)
        print()
    print("✅ Ferdig med reset!")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 13, Finished, Available, Finished, False)

8️⃣  RESET TIL DEFAULT

⏭️  Hoppet over – table_list_default er tom
💡 Sett table_list_default = ['12345', '67890'] for å resette


In [12]:
# ============================================================================
# 9️⃣  BATCH INSERT (legg til flere tabeller)
# ============================================================================

print("=" * 80)
print("9️⃣  BATCH INSERT")
print("=" * 80)

if not table_list:
    print("\n⏭️  Hoppet over – table_list er tom")
    print("💡 Sett table_list = ['12345', '67890'] for å legge til flere tabeller")
else:
    print(f"\n📋 Legger til {len(table_list)} tabeller...\n")
    ok = skipped = 0
    for tid in table_list:
        print(f"--- {tid} ---")
        result = add_new_table(
            tid,
            override_category=kategori,
            override_lookback=lookback,
            override_priority=priority_ny,
        )
        if result:
            ok += 1
        else:
            skipped += 1
        print()

    print(f"📊 Resultat: {ok} lagt til, {skipped} hoppet over")

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 14, Finished, Available, Finished, False)

9️⃣  BATCH INSERT

⏭️  Hoppet over – table_list er tom
💡 Sett table_list = ['12345', '67890'] for å legge til flere tabeller


In [13]:
# ============================================================================
# 🔟 RESET ALLE TABELLER TIL DEFAULT
# ============================================================================

print("=" * 80)
print("🔟 RESET ALLE TABELLER TIL DEFAULT")
print("=" * 80)

print("""
⚠️  ADVARSEL: Resetter kategori og lookback_periods for ALLE tabeller!
   last_loaded_timestamp og priority røres IKKE.

For å kjøre:
   reset_all_tables_to_defaults(confirm=True)
""")

# Kommenter inn for å kjøre:
# reset_all_tables_to_defaults(confirm=True)

StatementMeta(, 1023aacb-1f4d-4dd7-98c1-551ebee04142, 15, Finished, Available, Finished, False)

🔟 RESET ALLE TABELLER TIL DEFAULT

⚠️  ADVARSEL: Resetter kategori og lookback_periods for ALLE tabeller!
   last_loaded_timestamp og priority røres IKKE.

For å kjøre:
   reset_all_tables_to_defaults(confirm=True)

